# Evaluate prediction exports

Run `evaluate.py` first (see README). This notebook consumes saved predictions; training data and checkpoints are not needed for plotting. Run from the repository root.


In [ ]:
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc

PREDICTIONS = Path("outputs/validation_predictions.h5")
TARGET_FPR = 0.00054


In [ ]:
with h5py.File(PREDICTIONS, "r") as data:
    targets = data["event_targets"][:]
    probabilities = data["event_probabilities"][:]
    predictions = data["event_predictions"][:]

class_ids = np.arange(probabilities.shape[1])
# These names match event_current_targets=True; adjust for other label encodings.
class_names = ["other", "mu", "e", "NNbar"]
if len(class_names) != len(class_ids):
    class_names = [str(i) for i in class_ids]


## One-vs-rest ROC

The marked point preserves the training metric convention: the first ROC point at or above the requested FPR. Its actual FPR can be higher. Classes missing positive or negative examples are skipped.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for class_id, name in zip(class_ids, class_names):
    binary_targets = targets == class_id
    if binary_targets.all() or not binary_targets.any():
        print(f"Skipping {name}: only one target value is present")
        continue
    fpr, tpr, _ = roc_curve(binary_targets, probabilities[:, class_id], drop_intermediate=False)
    index = min(np.searchsorted(fpr, TARGET_FPR), len(fpr) - 1)
    ax.plot(fpr, tpr, label=f"{name}: AUC={auc(fpr, tpr):.4f}")
    ax.plot(fpr[index], tpr[index], "o")
    print(f"{name}: actual FPR={fpr[index]:.6%}, TPR={tpr[index]:.4%}")
ax.axvline(TARGET_FPR, color="grey", linestyle="--")
ax.set(xlabel="False positive rate", ylabel="True positive rate")
ax.legend()
plt.show()


## Confusion matrix


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    targets, predictions, labels=class_ids, display_labels=class_names
)
plt.show()


## NNbar score distribution

The last class is treated as NNbar, matching training.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for class_id, name in zip(class_ids, class_names):
    scores = probabilities[targets == class_id, -1]
    if scores.size:
        ax.hist(scores, bins=np.linspace(0, 1, 51), histtype="step", label=name)
ax.set(xlabel="NNbar probability", ylabel="Events", yscale="log")
ax.legend()
plt.show()
